# Étape 3 - Chunking avant indexation FAISS

---

- **Projet 9 :** Concevez et déployez un système RAG
- **Application :** Écho - Chatbot culturel
- **Auteur :** Justine Tranchant
- **Date :** mai 2026

---

Objectif : comparer plusieurs stratégies de chunking avant de justifier la stratégie retenue pour le pipeline FAISS.


# Imports

On charge les documents OpenAgenda déjà construits, la fonction principale de chunking du projet et `spaCy` pour la comparaison par phrases.


In [1]:
import json
import sys

import pandas as pd
import spacy

sys.path.append("..")

from src.chunking import build_chunks
from src.config import (
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    PATHS,
    PROCESSED_EVENTS_DOCUMENTS_FILENAME,
)


In [2]:
documents_path = PATHS.data_processed / PROCESSED_EVENTS_DOCUMENTS_FILENAME

with documents_path.open("r", encoding="utf-8") as f:
    documents = [json.loads(line) for line in f if line.strip()]

print(f"Fichier chargé : data/processed/{documents_path.name}")
print(f"Nombre de documents : {len(documents)}")


Fichier chargé : data/processed/events_documents.jsonl
Nombre de documents : 138


In [3]:
document_lengths = [len(document.get("document_text") or "") for document in documents]
documents_over_chunk_size = sum(1 for length in document_lengths if length > CHUNK_SIZE)

document_length_stats = {
    "total_documents": len(documents),
    "min_length": min(document_lengths),
    "median_length": sorted(document_lengths)[len(document_lengths) // 2],
    "max_length": max(document_lengths),
    "mean_length": round(sum(document_lengths) / len(document_lengths), 1),
    "documents_over_chunk_size": documents_over_chunk_size,
    "ratio_over_chunk_size": round(documents_over_chunk_size / len(documents), 3),
}

print(document_length_stats)


{'total_documents': 138, 'min_length': 375, 'median_length': 948, 'max_length': 5411, 'mean_length': 967.4, 'documents_over_chunk_size': 9, 'ratio_over_chunk_size': 0.065}


# Chargement du modèle spaCy

Cette comparaison utilise réellement `fr_core_news_sm` pour segmenter les documents en phrases.


In [4]:
nlp = spacy.load("fr_core_news_sm")
print(f"spaCy version : {spacy.__version__}")
print("Modèle chargé : fr_core_news_sm")


spaCy version : 3.8.14
Modèle chargé : fr_core_news_sm


# Comparaison des stratégies de chunking

On compare ici quatre stratégies simples :

1. **Sans chunking** : un document OpenAgenda devient un seul chunk.
2. **Taille + overlap** : on réutilise la fonction principale du projet.
3. **Markdown simple** : on découpe les documents en suivant les titres `#` et `##`.
4. **spaCy phrases** : on découpe les documents en phrases, puis on regroupe ces phrases en chunks d'environ `CHUNK_SIZE`.


In [5]:
def split_markdown_blocks(text):
    blocks = []
    current_lines = []
    for line in text.splitlines():
        if line.strip() == "":
            if current_lines:
                blocks.append("\n".join(current_lines).strip())
                current_lines = []
            continue
        current_lines.append(line)
    if current_lines:
        blocks.append("\n".join(current_lines).strip())
    return [block for block in blocks if block.strip()]


def build_no_chunk_strategy(documents):
    chunks = []
    for document in documents:
        event_id = str(document.get("event_id") or "").strip()
        text = str(document.get("document_text") or "").strip()
        metadata = document.get("metadata") if isinstance(document.get("metadata"), dict) else {}
        if not event_id or not text:
            continue
        chunks.append(
            {
                "chunk_id": f"{event_id}_0",
                "event_id": event_id,
                "chunk_index": 0,
                "chunk_count": 1,
                "chunk_text": text,
                "metadata": metadata.copy(),
            }
        )
    return chunks


def chunk_markdown_document(document):
    text = str(document.get("document_text") or "").strip()
    if not text:
        return []
    lines = text.splitlines()
    title = lines[0].strip() if lines and lines[0].strip().startswith("# ") else ""
    remaining_lines = lines[1:] if title else lines
    sections = []
    current_section = []
    for line in remaining_lines:
        stripped = line.strip()
        if stripped.startswith("# ") or stripped.startswith("## "):
            if current_section:
                section_text = "\n".join(current_section).strip()
                if section_text:
                    sections.append(section_text)
            current_section = [stripped]
            continue
        if current_section or stripped:
            current_section.append(line)
    if current_section:
        section_text = "\n".join(current_section).strip()
        if section_text:
            sections.append(section_text)
    chunks = []
    for section in sections:
        chunk_text = section
        if title and not section.startswith(title):
            chunk_text = f"{title}\n\n{section}"
        chunk_text = chunk_text.strip()
        if chunk_text:
            chunks.append(chunk_text)
    return chunks or [text]


def build_markdown_chunks(documents):
    chunks = []
    for document in documents:
        event_id = str(document.get("event_id") or "").strip()
        metadata = document.get("metadata") if isinstance(document.get("metadata"), dict) else {}
        if not event_id:
            continue
        parts = chunk_markdown_document(document)
        if not parts:
            continue
        for index, part in enumerate(parts):
            chunks.append(
                {
                    "chunk_id": f"{event_id}_{index}",
                    "event_id": event_id,
                    "chunk_index": index,
                    "chunk_count": len(parts),
                    "chunk_text": part,
                    "metadata": metadata.copy(),
                }
            )
    return chunks


def split_block_with_spacy(block, nlp):
    stripped_block = block.strip()
    if not stripped_block:
        return []
    if stripped_block.startswith("#") or stripped_block.startswith("- "):
        return [stripped_block]
    if "\n- " in stripped_block:
        return [stripped_block]
    if len(stripped_block) <= 200:
        return [stripped_block]
    doc = nlp(stripped_block)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    return sentences or [stripped_block]


def chunk_spacy_document(document, nlp, chunk_size=CHUNK_SIZE):
    text = str(document.get("document_text") or "").strip()
    if not text:
        return []
    lines = text.splitlines()
    title = lines[0].strip() if lines and lines[0].strip().startswith("# ") else ""
    body = "\n".join(lines[1:]).strip() if title else text
    if not body:
        return [text]
    blocks = split_markdown_blocks(body)
    units = []
    for block in blocks:
        units.extend(split_block_with_spacy(block, nlp))
    if not units:
        return [text]
    chunks = []
    current_parts = []
    current_length = len(title) + 2 if title else 0
    for unit in units:
        separator_length = 2 if current_parts else 0
        projected_length = current_length + separator_length + len(unit)
        if current_parts and projected_length > chunk_size:
            chunk_body = "\n\n".join(current_parts).strip()
            chunk_text = f"{title}\n\n{chunk_body}".strip() if title else chunk_body
            if chunk_text:
                chunks.append(chunk_text)
            current_parts = [unit]
            current_length = len(title) + 2 + len(unit) if title else len(unit)
            continue
        current_parts.append(unit)
        current_length = projected_length
    if current_parts:
        chunk_body = "\n\n".join(current_parts).strip()
        chunk_text = f"{title}\n\n{chunk_body}".strip() if title else chunk_body
        if chunk_text:
            chunks.append(chunk_text)
    return chunks or [text]


def build_spacy_chunks(documents, nlp):
    chunks = []
    for document in documents:
        event_id = str(document.get("event_id") or "").strip()
        metadata = document.get("metadata") if isinstance(document.get("metadata"), dict) else {}
        if not event_id:
            continue
        parts = chunk_spacy_document(document, nlp)
        if not parts:
            continue
        for index, part in enumerate(parts):
            chunks.append(
                {
                    "chunk_id": f"{event_id}_{index}",
                    "event_id": event_id,
                    "chunk_index": index,
                    "chunk_count": len(parts),
                    "chunk_text": part,
                    "metadata": metadata.copy(),
                }
            )
    return chunks


def summarize_strategy(name, chunks):
    lengths = [len(chunk["chunk_text"]) for chunk in chunks]
    event_ids = {chunk["event_id"] for chunk in chunks}
    return {
        "Stratégie": name,
        "Chunks totaux": len(chunks),
        "Chunks moyens / événement": round(len(chunks) / len(event_ids), 2),
        "Longueur moyenne": round(sum(lengths) / len(lengths), 1),
        "Longueur maximale": max(lengths),
    }


def build_preview_text(strategy_name, chunks, event_id):
    separator = "----" * 20
    selected_chunks = [chunk for chunk in chunks if chunk["event_id"] == event_id]
    print(strategy_name)
    print(f"Nombre de chunks : {len(selected_chunks)}")
    print(separator)

    preview_chunks = selected_chunks[:2]
    for index, chunk in enumerate(preview_chunks):
        print(f"- {chunk['chunk_id']} ({len(chunk['chunk_text'])} caractères)")
        print(chunk["chunk_text"].rstrip())
        if index < len(preview_chunks) - 1:
            print()
            print(separator)


In [15]:
no_chunk_chunks = build_no_chunk_strategy(documents)
size_chunks = build_chunks(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
markdown_chunks = build_markdown_chunks(documents)
spacy_chunks = build_spacy_chunks(documents, nlp)

comparison_df = pd.DataFrame(
    [
        summarize_strategy("Sans chunking", no_chunk_chunks),
        summarize_strategy("Taille + overlap", size_chunks),
        summarize_strategy("Markdown simple", markdown_chunks),
        summarize_strategy("spaCy phrases", spacy_chunks),
    ]
)

comparison_df

,Stratégie,Chunks totaux,Chunks moyens / événement,Longueur moyenne,Longueur maximale
0,Sans chunking,138,1.00,967.4,5411
1,Taille + overlap,155,1.12,880.1,1207
2,Markdown simple,414,3.00,372.4,5153
3,spaCy phrases,152,1.10,882.4,1197


# Exemples comparatifs

## Exemple 1 - Événement court

Événement utilisé : `10772673` - **Mai à Vélo 2026**


### Sans chunking


In [7]:
build_preview_text("Sans chunking", no_chunk_chunks, "10772673")

Sans chunking
Nombre de chunks : 1
--------------------------------------------------------------------------------
- 10772673_0 (375 caractères)
# Mai à Vélo 2026

## Description
Faites parcourir à votre communauté le plus de kilomètres au mois de mai!

## Informations pratiques
- Lieu : mios
- Ville : Mios
- Adresse : 3 Avenue de la République, 33380 Mios, France
- Dates : du 2026-04-30 22:00 au 2026-05-31 21:59
- Mode de participation : Sur place
- Statut : Programmé

## Source
- Organisateur : Challenges Geovelo


### Taille + overlap


In [8]:
build_preview_text("Taille + overlap", size_chunks, "10772673")

Taille + overlap
Nombre de chunks : 1
--------------------------------------------------------------------------------
- 10772673_0 (375 caractères)
# Mai à Vélo 2026

## Description
Faites parcourir à votre communauté le plus de kilomètres au mois de mai!

## Informations pratiques
- Lieu : mios
- Ville : Mios
- Adresse : 3 Avenue de la République, 33380 Mios, France
- Dates : du 2026-04-30 22:00 au 2026-05-31 21:59
- Mode de participation : Sur place
- Statut : Programmé

## Source
- Organisateur : Challenges Geovelo


### Markdown simple


In [9]:
build_preview_text("Markdown simple", markdown_chunks, "10772673")

Markdown simple
Nombre de chunks : 3
--------------------------------------------------------------------------------
- 10772673_0 (107 caractères)
# Mai à Vélo 2026

## Description
Faites parcourir à votre communauté le plus de kilomètres au mois de mai!

--------------------------------------------------------------------------------
- 10772673_1 (238 caractères)
# Mai à Vélo 2026

## Informations pratiques
- Lieu : mios
- Ville : Mios
- Adresse : 3 Avenue de la République, 33380 Mios, France
- Dates : du 2026-04-30 22:00 au 2026-05-31 21:59
- Mode de participation : Sur place
- Statut : Programmé


### spaCy phrases


In [10]:
build_preview_text("spaCy phrases", spacy_chunks, "10772673")

spaCy phrases
Nombre de chunks : 1
--------------------------------------------------------------------------------
- 10772673_0 (375 caractères)
# Mai à Vélo 2026

## Description
Faites parcourir à votre communauté le plus de kilomètres au mois de mai!

## Informations pratiques
- Lieu : mios
- Ville : Mios
- Adresse : 3 Avenue de la République, 33380 Mios, France
- Dates : du 2026-04-30 22:00 au 2026-05-31 21:59
- Mode de participation : Sur place
- Statut : Programmé

## Source
- Organisateur : Challenges Geovelo


## Exemple 2 - Événement structuré plus long

Événement utilisé : `13708573` - **Initiation à l'astronomie à Lanton**


### Sans chunking


In [11]:
build_preview_text("Sans chunking", no_chunk_chunks, "13708573")

Sans chunking
Nombre de chunks : 1
--------------------------------------------------------------------------------
- 13708573_0 (1814 caractères)
# Initiation à l'astronomie à Lanton

## Description
Initiation à l'astronomie et observation aux instruments.

✨ Découvrez le ciel nocturne comme vous ne l’avez jamais observé 🌙🔭

Lors de cette soirée d’observation du ciel, laissez-vous émerveiller par les beautés cachées de l’espace, visibles uniquement grâce aux télescopes et aux jumelles astronomiques.
Planètes, galaxies, nébuleuses… vous découvrirez les plus beaux objets célestes du moment.

Tout au long de la soirée, vous comprendrez :

- le parcours de vie d’une étoile, de sa naissance à sa fin
- les grands principes de la mécanique céleste et le déplacement apparent des astres

Vous apprendrez également à :

- reconnaître plusieurs constellations, et peut-être celle de votre signe zodiacal si elle est visible
- découvrir les mythes et légendes associés aux constellations

Des expérie

### Taille + overlap


In [12]:
build_preview_text("Taille + overlap", size_chunks, "13708573")

Taille + overlap
Nombre de chunks : 2
--------------------------------------------------------------------------------
- 13708573_0 (1143 caractères)
# Initiation à l'astronomie à Lanton

## Description
Initiation à l'astronomie et observation aux instruments.

✨ Découvrez le ciel nocturne comme vous ne l’avez jamais observé 🌙🔭

Lors de cette soirée d’observation du ciel, laissez-vous émerveiller par les beautés cachées de l’espace, visibles uniquement grâce aux télescopes et aux jumelles astronomiques.
Planètes, galaxies, nébuleuses… vous découvrirez les plus beaux objets célestes du moment.

Tout au long de la soirée, vous comprendrez :

- le parcours de vie d’une étoile, de sa naissance à sa fin
- les grands principes de la mécanique céleste et le déplacement apparent des astres

Vous apprendrez également à :

- reconnaître plusieurs constellations, et peut-être celle de votre signe zodiacal si elle est visible
- découvrir les mythes et légendes associés aux constellations

Des expé

### Markdown simple


In [13]:
build_preview_text("Markdown simple", markdown_chunks, "13708573")

Markdown simple
Nombre de chunks : 3
--------------------------------------------------------------------------------
- 13708573_0 (1498 caractères)
# Initiation à l'astronomie à Lanton

## Description
Initiation à l'astronomie et observation aux instruments.

✨ Découvrez le ciel nocturne comme vous ne l’avez jamais observé 🌙🔭

Lors de cette soirée d’observation du ciel, laissez-vous émerveiller par les beautés cachées de l’espace, visibles uniquement grâce aux télescopes et aux jumelles astronomiques.
Planètes, galaxies, nébuleuses… vous découvrirez les plus beaux objets célestes du moment.

Tout au long de la soirée, vous comprendrez :

- le parcours de vie d’une étoile, de sa naissance à sa fin
- les grands principes de la mécanique céleste et le déplacement apparent des astres

Vous apprendrez également à :

- reconnaître plusieurs constellations, et peut-être celle de votre signe zodiacal si elle est visible
- découvrir les mythes et légendes associés aux constellations

Des expér

### spaCy phrases


In [14]:
build_preview_text("spaCy phrases", spacy_chunks, "13708573")

spaCy phrases
Nombre de chunks : 2
--------------------------------------------------------------------------------
- 13708573_0 (1144 caractères)
# Initiation à l'astronomie à Lanton

## Description
Initiation à l'astronomie et observation aux instruments.

✨ Découvrez le ciel nocturne comme vous ne l’avez jamais observé 🌙🔭

Lors de cette soirée d’observation du ciel, laissez-vous émerveiller par les beautés cachées de l’espace, visibles uniquement grâce aux télescopes et aux jumelles astronomiques.
Planètes, galaxies, nébuleuses…

vous découvrirez les plus beaux objets célestes du moment.

Tout au long de la soirée, vous comprendrez :

- le parcours de vie d’une étoile, de sa naissance à sa fin
- les grands principes de la mécanique céleste et le déplacement apparent des astres

Vous apprendrez également à :

- reconnaître plusieurs constellations, et peut-être celle de votre signe zodiacal si elle est visible
- découvrir les mythes et légendes associés aux constellations

Des expéri